In [ ]:
!pip install xgboost
!pip install imblearn

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.metrics import roc_auc_score, classification_report, brier_score_loss, log_loss, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV

## Models

In [ ]:
def stacking_model(X_train, y_train, X_test, y_test):
  # Define preprocessor
  preprocessor = ColumnTransformer(
      transformers=[
          ('cat', OneHotEncoder(handle_unknown='ignore'), [0]),
          ('num', StandardScaler(), slice(1, None))
      ]
  )

  # Define the Base Models inside an Imbalanced Pipeline
  # Every model gets its data preprocessed and balanced via SMOTE independently inside the CV folds
  def build_pipeline(classifier):
      return ImbPipeline(steps=[
          ('preprocessor', preprocessor),
          ('smote', SMOTE(random_state=42, sampling_strategy=0.5)),
          ('clf', classifier)
      ])

  estimators = [
      ('xgb', build_pipeline(
          XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.05, random_state=42)
      )),

      ('nn', build_pipeline(
          MLPClassifier(hidden_layer_sizes=(13, 8), activation='relu', max_iter=500, random_state=42)
      )),

      ('gpc', build_pipeline(
          GaussianProcessClassifier(kernel=1.0 * RBF(1.0), random_state=42, n_jobs=-1)
      )),

      ('svc', build_pipeline(
          SVC(kernel='rbf', probability=True, C=0.1, random_state=42)
      ))
  ]

  # 3. Define the Meta-Model
  # class_weight='balanced' ensures the Random Forest respects the minority class
  meta_model = RandomForestClassifier(n_estimators=100, max_depth=3, class_weight='balanced', random_state=42)

  # Build the Stacking Ensemble
  stacking_clf = StackingClassifier(
      estimators=estimators,
      final_estimator=meta_model,
      cv=5,
      passthrough=False,
      n_jobs=-1
  )

  # Train and Predict
  stacking_clf.fit(X_train, y_train)
  predictions = stacking_clf.predict(X_test)
  probabilities = stacking_clf.predict_proba(X_test)[:, 1]

  print(f"\n--- Stacking Ensemble Performance ---")
  print(f"Log Loss: {log_loss(y_test, probabilities):.4f}")
  print(f"Brier Score: {brier_score_loss(y_test, probabilities):.4f}")
  print(f"AUC: {roc_auc_score(y_test, probabilities):.4f}")
  print(f"F1-Score: {f1_score(y_test, predictions):.4f}")
  print("\nClassification Report:")
  print(classification_report(y_test, predictions, digits=4))

In [ ]:
def calibrated_model(X_train, y_train, X_test, y_test):
  # Define the Preprocessor
  preprocessor = ColumnTransformer(
      transformers=[
          ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), [0])
      ],
      remainder='passthrough'
  )

  # Define the Base Model
  xgb = XGBClassifier(
      n_estimators=100,
      max_depth=3,
      learning_rate=0.05,
      random_state=42
  )

  # Bind them together in a Pipeline
  model_pipeline = Pipeline(steps=[
      ('preprocessor', preprocessor),
      ('classifier', xgb)
  ])

  # Calibrate the entire Pipeline
  # This ensures both the encoding and the model predictions are perfectly calibrated
  calibrated_model = CalibratedClassifierCV(model_pipeline, method='isotonic', cv=5)

  # Train and Predict
  calibrated_model.fit(X_train, y_train)
  goal_probabilities = calibrated_model.predict_proba(X_test)[:, 1]
  predictions = calibrated_model.predict(X_test)

  print(f"\n--- Calibrated Model Performance ---")
  print(f"Log Loss: {log_loss(y_test, goal_probabilities):.4f}")
  print(f"Brier Score: {brier_score_loss(y_test, goal_probabilities):.4f}")
  print(f"AUC: {roc_auc_score(y_test, goal_probabilities):.4f}")
  print(f"F1-Score: {f1_score(y_test, predictions):.4f}")
  print("\nClassification Report:")
  print(classification_report(y_test, predictions, digits=4))

In [ ]:
def xbgclass(X_train, y_train, X_test, y_test):
  # Define preprocessor and pipeline
  preprocessor = ColumnTransformer(
      transformers=[
          ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['position'])
      ],
      remainder='passthrough'
  )

  model_pipeline = Pipeline(steps=[
      ('preprocessor', preprocessor),
      ('classifier', XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.05, random_state=42))
  ])

  # Define grid and search
  param_grid = {
      'classifier__scale_pos_weight': list(range(10, 21))
  }

  inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

  grid_search = GridSearchCV(
      estimator=model_pipeline,
      param_grid=param_grid,
      cv=inner_cv,
      scoring=['roc_auc', 'f1'],
      refit='roc_auc',
      n_jobs=-1
  )

  # Fit the optimal model
  print("Hunting for optimal scale_pos_weight and fitting final model...")
  grid_search.fit(X_train, y_train)

  # Extract the optimal weight it found
  optimal_weight = grid_search.best_params_['classifier__scale_pos_weight']
  best_cv_f1 = grid_search.best_score_
  print(f"\n✅ Optimal scale_pos_weight found: {optimal_weight}")
  print(f"✅ Expected F1-Score (from CV): {best_cv_f1:.4f}")

  # Predict
  print("\nMaking predictions on the unseen test set...")

  y_pred = grid_search.predict(X_test)
  y_proba = grid_search.predict_proba(X_test)[:, 1]

  # Evaluate the predictions
  final_f1 = f1_score(y_test, y_pred)
  final_auc = roc_auc_score(y_test, y_proba)

  print(f"\n--- XGBClassifier Performance ---")
  print(f"Log Loss: {log_loss(y_test, y_test):.4f}")
  print(f"Brier Score: {brier_score_loss(y_test, y_proba):.4f}")
  print(f"AUC: {final_auc:.4f}")
  print(f"F1-Score: {final_f1:.4f}")
  print("\nClassification Report:")
  print(classification_report(y_test, y_pred, digits=4))

## Full augmented data frame

In [ ]:
df = pd.read_csv("very_final_dataset.csv")
X = df.iloc[:,:-1]
y = df.iloc[:,-1]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [ ]:
stacking_model(X_train, y_train, X_test, y_test)
calibrated_model(X_train, y_train, X_test, y_test)
xbgclass(X_train, y_train, X_test, y_test)


--- Stacking Ensemble Performance ---
Log Loss: 0.5681
Brier Score: 0.1948
AUC: 0.7119
F1-Score: 0.1742

Classification Report:
              precision    recall  f1-score   support

           0     0.9621    0.6956    0.8074       657
           1     0.1031    0.5610    0.1742        41

    accuracy                         0.6877       698
   macro avg     0.5326    0.6283    0.4908       698
weighted avg     0.9117    0.6877    0.7702       698


--- Calibrated Model Performance ---
Log Loss: 0.2068
Brier Score: 0.0539
AUC: 0.7205
F1-Score: 0.0000

Classification Report:
              precision    recall  f1-score   support

           0     0.9413    1.0000    0.9697       657
           1     0.0000    0.0000    0.0000        41

    accuracy                         0.9413       698
   macro avg     0.4706    0.5000    0.4849       698
weighted avg     0.8860    0.9413    0.9128       698

Hunting for optimal scale_pos_weight and fitting final model...


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



✅ Optimal scale_pos_weight found: 12
✅ Expected F1-Score (from CV): 0.7374

Making predictions on the unseen test set...

--- XGBClassifier Performance ---
Log Loss: 0.0000
Brier Score: 0.1312
AUC: 0.7398
F1-Score: 0.2192

Classification Report:
              precision    recall  f1-score   support

           0     0.9578    0.8645    0.9088       657
           1     0.1524    0.3902    0.2192        41

    accuracy                         0.8367       698
   macro avg     0.5551    0.6274    0.5640       698
weighted avg     0.9105    0.8367    0.8683       698



## Full augmented dataframe without goalkeepers

In [ ]:
df = pd.read_csv("very_final_dataset.csv")
df = df[df['position'] != 'G']
X = df.iloc[:,:-1]
y = df.iloc[:,-1]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [ ]:
stacking_model(X_train, y_train, X_test, y_test)
calibrated_model(X_train, y_train, X_test, y_test)
xbgclass(X_train, y_train, X_test, y_test)


--- Stacking Ensemble Performance ---
Log Loss: 0.5829
Brier Score: 0.1978
AUC: 0.6773
F1-Score: 0.1754

Classification Report:
              precision    recall  f1-score   support

           0     0.9530    0.7184    0.8192       593
           1     0.1070    0.4878    0.1754        41

    accuracy                         0.7035       634
   macro avg     0.5300    0.6031    0.4973       634
weighted avg     0.8983    0.7035    0.7776       634


--- Calibrated Model Performance ---
Log Loss: 0.2115
Brier Score: 0.0563
AUC: 0.7501
F1-Score: 0.0000

Classification Report:
              precision    recall  f1-score   support

           0     0.9353    1.0000    0.9666       593
           1     0.0000    0.0000    0.0000        41

    accuracy                         0.9353       634
   macro avg     0.4677    0.5000    0.4833       634
weighted avg     0.8748    0.9353    0.9041       634

Hunting for optimal scale_pos_weight and fitting final model...


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



✅ Optimal scale_pos_weight found: 11
✅ Expected F1-Score (from CV): 0.7336

Making predictions on the unseen test set...

--- XGBClassifier Performance ---
Log Loss: 0.0000
Brier Score: 0.1318
AUC: 0.7444
F1-Score: 0.2171

Classification Report:
              precision    recall  f1-score   support

           0     0.9505    0.8752    0.9113       593
           1     0.1591    0.3415    0.2171        41

    accuracy                         0.8407       634
   macro avg     0.5548    0.6083    0.5642       634
weighted avg     0.8994    0.8407    0.8664       634



## Last 15 dataset

In [ ]:
df = pd.read_csv("very_final_dataset.csv")

first_two = df.columns[:2].tolist()
last15_cols = df.columns[df.columns.str.startswith('last15_')].tolist()
last_one = df.columns[-1:].tolist()

cols_to_keep = list(dict.fromkeys(first_two + last15_cols + last_one))

df = df[cols_to_keep]

X = df.iloc[:,:-1]
y = df.iloc[:,-1]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [ ]:
stacking_model(X_train, y_train, X_test, y_test)
calibrated_model(X_train, y_train, X_test, y_test)
xbgclass(X_train, y_train, X_test, y_test)


--- Stacking Ensemble Performance ---
Log Loss: 0.5819
Brier Score: 0.2019
AUC: 0.7014
F1-Score: 0.1811

Classification Report:
              precision    recall  f1-score   support

           0     0.9617    0.7260    0.8274       657
           1     0.1089    0.5366    0.1811        41

    accuracy                         0.7149       698
   macro avg     0.5353    0.6313    0.5042       698
weighted avg     0.9116    0.7149    0.7894       698


--- Calibrated Model Performance ---
Log Loss: 0.2138
Brier Score: 0.0554
AUC: 0.6809
F1-Score: 0.0000

Classification Report:
              precision    recall  f1-score   support

           0     0.9412    0.9985    0.9690       657
           1     0.0000    0.0000    0.0000        41

    accuracy                         0.9398       698
   macro avg     0.4706    0.4992    0.4845       698
weighted avg     0.8859    0.9398    0.9121       698

Hunting for optimal scale_pos_weight and fitting final model...

✅ Optimal scale_pos_weig

## Last 15 dataset without goalkeepers

In [ ]:
df = pd.read_csv("very_final_dataset.csv")
df = df[df['position'] != 'G']

first_two = df.columns[:2].tolist()
last15_cols = df.columns[df.columns.str.startswith('last15_')].tolist()
last_one = df.columns[-1:].tolist()

cols_to_keep = list(dict.fromkeys(first_two + last15_cols + last_one))

df = df[cols_to_keep]

X = df.iloc[:,:-1]
y = df.iloc[:,-1]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [ ]:
stacking_model(X_train, y_train, X_test, y_test)
calibrated_model(X_train, y_train, X_test, y_test)
xbgclass(X_train, y_train, X_test, y_test)


--- Stacking Ensemble Performance ---
Log Loss: 0.6067
Brier Score: 0.2083
AUC: 0.6486
F1-Score: 0.1653

Classification Report:
              precision    recall  f1-score   support

           0     0.9515    0.6948    0.8031       593
           1     0.0995    0.4878    0.1653        41

    accuracy                         0.6814       634
   macro avg     0.5255    0.5913    0.4842       634
weighted avg     0.8964    0.6814    0.7619       634


--- Calibrated Model Performance ---
Log Loss: 0.2233
Brier Score: 0.0583
AUC: 0.6993
F1-Score: 0.0000

Classification Report:
              precision    recall  f1-score   support

           0     0.9353    1.0000    0.9666       593
           1     0.0000    0.0000    0.0000        41

    accuracy                         0.9353       634
   macro avg     0.4677    0.5000    0.4833       634
weighted avg     0.8748    0.9353    0.9041       634

Hunting for optimal scale_pos_weight and fitting final model...


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



✅ Optimal scale_pos_weight found: 13
✅ Expected F1-Score (from CV): 0.6219

Making predictions on the unseen test set...

--- XGBClassifier Performance ---
Log Loss: 0.0000
Brier Score: 0.1613
AUC: 0.6803
F1-Score: 0.1899

Classification Report:
              precision    recall  f1-score   support

           0     0.9497    0.8280    0.8847       593
           1     0.1282    0.3659    0.1899        41

    accuracy                         0.7981       634
   macro avg     0.5390    0.5969    0.5373       634
weighted avg     0.8966    0.7981    0.8398       634



## Cumul dataset

In [ ]:
df = pd.read_csv("very_final_dataset.csv")

first_two = df.columns[:2].tolist()
last15_cols = df.columns[df.columns.str.startswith('cumul_')].tolist()
last_one = df.columns[-1:].tolist()

cols_to_keep = list(dict.fromkeys(first_two + last15_cols + last_one))
df = df[cols_to_keep]

X = df.iloc[:,:-1]
y = df.iloc[:,-1]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [ ]:
stacking_model(X_train, y_train, X_test, y_test)
calibrated_model(X_train, y_train, X_test, y_test)
xbgclass(X_train, y_train, X_test, y_test)


--- Stacking Ensemble Performance ---
Log Loss: 0.5474
Brier Score: 0.1860
AUC: 0.7465
F1-Score: 0.1941

Classification Report:
              precision    recall  f1-score   support

           0     0.9641    0.7367    0.8352       657
           1     0.1173    0.5610    0.1941        41

    accuracy                         0.7264       698
   macro avg     0.5407    0.6488    0.5146       698
weighted avg     0.9144    0.7264    0.7975       698


--- Calibrated Model Performance ---
Log Loss: 0.2016
Brier Score: 0.0532
AUC: 0.7516
F1-Score: 0.0000

Classification Report:
              precision    recall  f1-score   support

           0     0.9413    1.0000    0.9697       657
           1     0.0000    0.0000    0.0000        41

    accuracy                         0.9413       698
   macro avg     0.4706    0.5000    0.4849       698
weighted avg     0.8860    0.9413    0.9128       698

Hunting for optimal scale_pos_weight and fitting final model...


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



✅ Optimal scale_pos_weight found: 15
✅ Expected F1-Score (from CV): 0.7627

Making predictions on the unseen test set...

--- XGBClassifier Performance ---
Log Loss: 0.0000
Brier Score: 0.1613
AUC: 0.7348
F1-Score: 0.2460

Classification Report:
              precision    recall  f1-score   support

           0     0.9674    0.8128    0.8834       657
           1     0.1575    0.5610    0.2460        41

    accuracy                         0.7980       698
   macro avg     0.5625    0.6869    0.5647       698
weighted avg     0.9198    0.7980    0.8459       698



## Cumul dataset without goalkeepers

In [ ]:
df = pd.read_csv("very_final_dataset.csv")
df = df[df['position'] != 'G']

first_two = df.columns[:2].tolist()
last15_cols = df.columns[df.columns.str.startswith('cumul_')].tolist()
last_one = df.columns[-1:].tolist()

cols_to_keep = list(dict.fromkeys(first_two + last15_cols + last_one))
df = df[cols_to_keep]

X = df.iloc[:,:-1]
y = df.iloc[:,-1]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [ ]:
stacking_model(X_train, y_train, X_test, y_test)
calibrated_model(X_train, y_train, X_test, y_test)
xbgclass(X_train, y_train, X_test, y_test)


--- Stacking Ensemble Performance ---
Log Loss: 0.5433
Brier Score: 0.1823
AUC: 0.7604
F1-Score: 0.2468

Classification Report:
              precision    recall  f1-score   support

           0     0.9727    0.7218    0.8287       593
           1     0.1495    0.7073    0.2468        41

    accuracy                         0.7208       634
   macro avg     0.5611    0.7145    0.5377       634
weighted avg     0.9195    0.7208    0.7910       634


--- Calibrated Model Performance ---
Log Loss: 0.2100
Brier Score: 0.0562
AUC: 0.7590
F1-Score: 0.0000

Classification Report:
              precision    recall  f1-score   support

           0     0.9352    0.9983    0.9657       593
           1     0.0000    0.0000    0.0000        41

    accuracy                         0.9338       634
   macro avg     0.4676    0.4992    0.4829       634
weighted avg     0.8747    0.9338    0.9033       634

Hunting for optimal scale_pos_weight and fitting final model...

✅ Optimal scale_pos_weig

## Original dataframe

In [ ]:
df = pd.read_csv("players_quarters_final.csv")
df = df.iloc[:, np.r_[7,10, 14:df.shape[1]]]
df["minute_in"] = (df["minute_in"]>1).astype(int)
X = df.iloc[:,:-1]
y = df.iloc[:,-1]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [ ]:
stacking_model(X_train, y_train, X_test, y_test)
calibrated_model(X_train, y_train, X_test, y_test)
xbgclass(X_train, y_train, X_test, y_test)


--- Stacking Ensemble Performance ---
Log Loss: 0.5821
Brier Score: 0.2046
AUC: 0.6803
F1-Score: 0.1793

Classification Report:
              precision    recall  f1-score   support

           0     0.9666    0.6606    0.7848       657
           1     0.1044    0.6341    0.1793        41

    accuracy                         0.6590       698
   macro avg     0.5355    0.6474    0.4821       698
weighted avg     0.9159    0.6590    0.7492       698


--- Calibrated Model Performance ---
Log Loss: 0.2122
Brier Score: 0.0549
AUC: 0.7052
F1-Score: 0.0000

Classification Report:
              precision    recall  f1-score   support

           0     0.9413    1.0000    0.9697       657
           1     0.0000    0.0000    0.0000        41

    accuracy                         0.9413       698
   macro avg     0.4706    0.5000    0.4849       698
weighted avg     0.8860    0.9413    0.9128       698

Hunting for optimal scale_pos_weight and fitting final model...


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



✅ Optimal scale_pos_weight found: 11
✅ Expected F1-Score (from CV): 0.7293

Making predictions on the unseen test set...

--- XGBClassifier Performance ---
Log Loss: 0.0000
Brier Score: 0.1344
AUC: 0.6919
F1-Score: 0.2143

Classification Report:
              precision    recall  f1-score   support

           0     0.9566    0.8721    0.9124       657
           1     0.1515    0.3659    0.2143        41

    accuracy                         0.8424       698
   macro avg     0.5541    0.6190    0.5634       698
weighted avg     0.9093    0.8424    0.8714       698

